# 01 · Baseline single-model QA  (Phase 1 MVP)

**Who Wants to Be a PoliMillionaire?** — the first answering pipeline, this notebook is.

What it does: load Qwen2.5-7B in 4-bit, wire the QA pipeline (classify -> prompt -> generate -> parse),
run it over a small hand-checked dev question set, and report **accuracy + latency**. The live game,
we touch NOT here — offline measurement only, so the 30s timer we never start by accident.

Pipeline parts, all from `src/` imported they are (built modularly, they were):
`TransformersEngine` · `QuestionClassifier` · `PromptBuilder.zero_shot_v1` · `QAPipeline` ·
`BenchmarkRunner` + `metrics`.

> **GPU needed.** Runtime ▸ Change runtime type ▸ T4 GPU, select you must.

## 1 · Setup — mount Drive, find the repo, install deps
Auto-located on Drive the repo is (by the marker `src/schemas.py`), so the exact folder nesting matters not.

In [ ]:
# Mounted, the Drive must be -- there the repo lives.
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os, sys, glob

# Under Drive, the repo root we hunt -- the marker file src/schemas.py, find it we must.
search_base = '/content/gdrive/MyDrive'
hits = glob.glob(os.path.join(search_base, '**', 'src', 'schemas.py'), recursive=True)
assert hits, ('Found, the repo was NOT under Drive. '
              'Upload the whole NLP repo folder to MyDrive, you must.')

# The shortest match, the true repo root it is (a stray copy, avoid we do).
roots = sorted({os.path.dirname(os.path.dirname(h)) for h in hits}, key=len)
REPO_ROOT = roots[0]
SRC = os.path.join(REPO_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Into the repo root, change directory we do -- relative paths and git, simpler they become.
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)
print('On sys.path:', SRC)

In [ ]:
# The Phase-1 inference stack, install we do -- light it stays (RAG deps, in Phase 4 they come).
!pip install -q 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.43.0' sentencepiece einops pyyaml pandas matplotlib
print('Installed, the Phase-1 dependencies are.')

In [ ]:
import torch

# A GPU, present it must be -- on CPU, a 7B model in 30s answer we cannot.
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING -- no GPU. Runtime ▸ Change runtime type ▸ T4 GPU, switch you must.')

## 2 · Load the run config
Into every run logged it is (D-007) — reproducible, the science stays.

In [ ]:
from config import RunConfig

# The base config, from YAML we load.
config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'base.yaml'))
print('run_id:', config.run_id)
print('model:', config.model.name, '|', config.model.quantization, '|', config.model.dtype)
print('latency budget:', config.latency_budget_s, 's')
print('prompt strategy:', config.prompt_strategy)

## 3 · Load + warm up the model
On a T4, ~1–2 minutes the 4-bit load takes (a one-time cost). Warmup pays the first-call latency
**before** any question is timed — so the 30s budget, a cold start never eats it.

In [ ]:
import time
from inference.engine import TransformersEngine

# Once, the model we load -- the cold-start cost, here we pay it.
t0 = time.perf_counter()
engine = TransformersEngine(
    model_name=config.model.name,
    quantization=config.model.quantization,
    dtype=config.model.dtype,
)
load_s = time.perf_counter() - t0
print(f'Model loaded in {load_s:.1f}s')

# Warm up -- the first-call kernels, compiled before any timed question they are.
t0 = time.perf_counter()
engine.warmup()
warm_s = time.perf_counter() - t0
print(f'Warmup in {warm_s:.1f}s')

# A smoke generation -- the answering path works, confirm we do.
t0 = time.perf_counter()
out = engine.generate('Reply with the single letter B and nothing else.', max_new_tokens=5)
gen_s = time.perf_counter() - t0
print(f'Smoke generate in {gen_s:.2f}s -> {out!r}')
print(f'tokens_in={engine.last_tokens_in}, tokens_out={engine.last_tokens_out}')

## 4 · Wire the pipeline + a single-question demo
Dependency injection (D-006): engine, classifier and prompt builder, injected they are.
RAG and tools, off for the baseline they stay (Phase 4 and Phase 3 turn them on).

In [ ]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder
from agent.pipeline import QAPipeline
from evaluation.dataset import load_questions

# The collaborators, injected into the pipeline they are (D-006).
classifier = QuestionClassifier()
prompt_builder = PromptBuilder(strategy=config.prompt_strategy)
pipeline = QAPipeline(
    engine=engine,
    prompt_builder=prompt_builder,
    classifier=classifier,
    retriever=None,   # Off for the baseline -- Phase 4 turns it on.
    tools=None,       # Off for the baseline -- Phase 3 turns it on.
    latency_budget_s=config.latency_budget_s,
)

# The dev question set, from disk loaded it is.
questions = load_questions(os.path.join(REPO_ROOT, 'data', 'dev_questions.jsonl'))
print(f'Loaded {len(questions)} dev questions.')

# One question, end to end through the pipeline we run -- the moving parts, inspect them we do.
demo_q = questions[0]
enriched = classifier.classify(demo_q)
print('--- The built prompt ---')
print(prompt_builder.build(enriched))
print()
pred = pipeline.answer(demo_q)
print('--- The prediction ---')
print('raw_output:', repr(pred.raw_output))
print('parsed answer:', pred.answer, '| gold:', demo_q.gold, '| confidence:', pred.confidence)
print('latency_s:', round(pred.latency_s, 2), '| tokens_out:', pred.tokens_out)

## 5 · Run the baseline benchmark
Over all dev questions the pipeline runs; one `EvalRecord` per question, to JSONL it is logged (D-007).
Crash-safe it is — each line flushed immediately, so a Colab disconnect nothing it loses.

In [ ]:
from evaluation.runner import BenchmarkRunner

# Into the repo the run logs we write -- persistent on Drive they stay.
runner = BenchmarkRunner(
    pipeline=pipeline,
    config=config,
    log_root=os.path.join(REPO_ROOT, 'experiments', 'runs'),
)
run_path = runner.run(questions)
print('Run written to:', run_path)

## 6 · Results — accuracy + latency
From the JSONL log all of it derives — re-analyse without re-running the model, we can.

In [ ]:
from evaluation.metrics import load_runs, accuracy_by, latency_summary

# The logged run, back into a DataFrame we read.
df = load_runs([run_path])

# Overall accuracy, the headline number it is.
known = df[df['correct'].notna()]
overall = known['correct'].astype(float).mean()
print(f'Overall accuracy: {overall:.1%}  (n={len(known)})')
print()
print('--- Accuracy by topic ---')
print(accuracy_by(df, 'topic').to_string(index=False))
print()
print('--- Accuracy by level ---')
print(accuracy_by(df, 'level').to_string(index=False))
print()
print('--- Latency summary (seconds) ---')
for k, v in latency_summary(df).items():
    print(f'{k}: {v}')

In [ ]:
import matplotlib.pyplot as plt

# Accuracy by topic, a bar chart it becomes -- per-topic strengths, the rubric asks for them.
topic_acc = accuracy_by(df, 'topic').sort_values('accuracy')
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].barh(topic_acc['topic'], topic_acc['accuracy'], color='steelblue')
ax[0].set_xlim(0, 1)
ax[0].set_xlabel('accuracy')
ax[0].set_title('Baseline accuracy by topic')

# Latency per question, a histogram it forms -- the 30s wall, a red line marks it.
ax[1].hist(df['latency_s'].dropna(), bins=12, color='darkorange', edgecolor='black')
ax[1].axvline(30.0, color='red', linestyle='--', label='30s budget')
ax[1].set_xlabel('latency (s)')
ax[1].set_ylabel('questions')
ax[1].set_title('Per-question latency')
ax[1].legend()
plt.tight_layout()
plt.show()

## 7 · Observations + next steps

_Fill these in after the run (they feed the investigation rubric):_

- **30s feasibility:** does median + p95 latency sit comfortably under 30s on a T4? (see the latency summary)
- **Per-topic strengths:** which of the 6 topics is strongest / weakest at baseline?
- **Overconfidence:** any wrong answers carrying high `confidence`? (a confidence-vs-correctness look)
- **Failure modes:** scan the `raw_output` of the misses — chatter the parser mishandled, or real knowledge gaps?

**Next (Phase 2 — prompt engineering):** add `few_shot_v1` and `cot_v1` to `PromptBuilder`, re-run the
same benchmark, and compare via `metrics`. Same pipeline seam, same logging — only the strategy changes.

> Note: the dev set is small (~23 hand-checked questions) and English-only — it measures the *pipeline*,
> not final leaderboard skill. Grow it before drawing strong conclusions.